# 第6章 一般化固有値問題と線形手法の広がり ― デモノートブック

講義ノート第6章は「一見ばらばらな線形手法が、$\boldsymbol{A}\boldsymbol{w}=\lambda\boldsymbol{B}\boldsymbol{w}$ という
一つの型に $\boldsymbol{A}$ と $\boldsymbol{B}$ を入れ替えて収まる」ことを主張する（本文 表6.1）。
このノートブックでは、その型を実際に `scipy.linalg.eigh(A, B)` で解きながら、
PCA・LDA・CCA・古典的 MDS・スペクトラルクラスタリング・PLS を順に走らせ、
$\boldsymbol{B}$ が特異になったときに何が起こるかも数値で確かめる。

## 目次

- [6.1 一般化固有値問題と白色化](#61)（本文 6.1 節、定理 6.2、命題 6.4、式 (6.1)(6.2)）
- [6.2 LDA と PCA は違う方向を選ぶ](#62)（本文 6.3 節、定理 6.12、例 6.15、図6.1）
- [6.3 正準相関分析と階数不足による退化](#63)（本文 6.2 節、定理 6.8、式 (6.7)）
- [6.4 古典的 MDS ＝ 双対 PCA](#64)（本文 6.4 節、定理 6.20、系 6.22、式 (6.11)(6.12)）
- [6.5 スペクトラルクラスタリング](#65)（本文 6.5 節、定理 6.29、命題 6.31、図6.2）
- [6.6 部分最小二乗法（PLS）](#66)（本文 注意 6.32、式 (6.28)）
- [演習](#ex) / [演習の解答](#sol)

データ行列は本講義の規約どおり**列がサンプル**（$\boldsymbol{X}\in\mathbb{R}^{d\times n}$）である。
`scikit-learn` は行がサンプルの規約なので、渡すときに `X.T` と転置する。
全セルを上から順に実行しておよそ 1 分である。

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

<a name="61"></a>
## 6.1 一般化固有値問題と白色化

本文 6.1 節。$\boldsymbol{A}\boldsymbol{w}=\lambda\boldsymbol{B}\boldsymbol{w}$（式 (6.1)）は、
$\boldsymbol{B}$ が正定値なら $\boldsymbol{u}=\boldsymbol{B}^{1/2}\boldsymbol{w}$、
$\boldsymbol{C}=\boldsymbol{B}^{-1/2}\boldsymbol{A}\boldsymbol{B}^{-1/2}$ という**白色化**で通常の固有値問題に化ける
（定理 6.2）。例 6.3 の $2\times2$ を、`eigh(A, B)` と白色化の二通りで解いて突き合わせる。

In [ ]:
from scipy.linalg import eigh

A = np.array([[2., 1.], [1., 2.]])
B = np.array([[2., 0.], [0., 1.]])
lam, W = eigh(A, B)                        # A w = lambda B w を直接解く
wb, Ub = np.linalg.eigh(B)
Bmh = Ub @ np.diag(wb**-0.5) @ Ub.T        # B^{-1/2}
mu, U = np.linalg.eigh(Bmh @ A @ Bmh)      # C = B^{-1/2} A B^{-1/2} の固有値問題

print("eigh(A,B) の固有値      :", np.round(lam, 6))
print("白色化して解いた固有値  :", np.round(mu, 6))
print("手計算 (3±sqrt(3))/2    :", np.round(np.sort([(3 - np.sqrt(3))/2,
                                                     (3 + np.sqrt(3))/2]), 6))
print("W^T B W（B-直交性）    :\n", np.round(W.T @ B @ W, 10))
print("固有ベクトルは符号を除いて一致:", np.allclose(np.abs(W), np.abs(Bmh @ U)))
w1 = W[:, 1] * np.sign(W[0, 1])
print(f"最大固有値の B-正規化固有ベクトル w1 = {np.round(w1, 4)}"
      f"  (w1^T B w1 = {w1 @ B @ w1:.6f})")

固有値はどちらの解き方でも $(0.633975,\ 2.366025)$ で、
手計算の $\frac{3\pm\sqrt3}{2}$ と一致する。$\boldsymbol{W}^\top\boldsymbol{B}\boldsymbol{W}=\boldsymbol{I}_2$ が
出力で確認でき（定理 6.2 の (3)、$\boldsymbol{B}$-直交性）、固有ベクトルも符号を除いて一致する。
一般化固有値問題を解くとは、$\boldsymbol{B}$ で測った長さを 1 に直してから
通常の固有値問題を解き、座標を戻すことである。

### $\boldsymbol{B}$ が特異なとき

$\boldsymbol{w}\in\operatorname{Ker}\boldsymbol{B}$ では分母が 0 になるので、一般化 Rayleigh 商
$\mathcal{R}(\boldsymbol{w})=\boldsymbol{w}^\top\boldsymbol{A}\boldsymbol{w}/\boldsymbol{w}^\top\boldsymbol{B}\boldsymbol{w}$（式 (6.2)）は
**そこで定義されない**。「$+\infty$ になる」と決め打ってはいけない、というのが命題 6.4 で、
$\boldsymbol{w}_0\in\operatorname{Ker}\boldsymbol{B}$ に近づく経路 $\boldsymbol{w}(t)=\boldsymbol{w}_0+t\boldsymbol{w}_1$ に沿って
(i) $+\infty$、(ii) $-\infty$、(iii) 経路依存の三通りが起こる。
例 6.5（$\boldsymbol{A}=\operatorname{diag}(2,1,-1)$、$\boldsymbol{B}=\operatorname{diag}(1,0,0)$）を走らせる。

In [ ]:
A2 = np.diag([2.0, 1.0, -1.0])
B2 = np.diag([1.0, 0.0, 0.0])              # 特異：Ker B = span{e2, e3}
e1, e2, e3 = np.eye(3)


def rayleigh(w, A_=A2, B_=B2):
    """一般化 Rayleigh 商 R(w) = w^T A w / w^T B w（式 (6.2)）。"""
    return float((w @ A_ @ w) / (w @ B_ @ w))


ts = np.array([1e-1, 1e-2, 1e-3, 1e-4])
paths = {
    "(i)   w0=e2,    A の二次形式 +1": (e2, e1),
    "(ii)  w0=e3,    A の二次形式 -1": (e3, e1),
    "(iii) w0=e2+e3, 経路 w1=e1":      (e2 + e3, e1),
    "(iii) w0=e2+e3, 経路 w1=e1+e2":   (e2 + e3, e1 + e2),
}
sing_vals = {}
print("命題 6.4：B が特異なときの R(w0 + t w1) の t→0 での挙動")
for name, (w0, w1) in paths.items():
    vals = [rayleigh(w0 + t * w1) for t in ts]
    sing_vals[name] = vals
    print(f"  {name:32s} w0^T A w0 = {w0 @ A2 @ w0:+.0f}  "
          + "  ".join(f"{v:+.4g}" for v in vals))

try:
    eigh(A2, B2)
except Exception as exc:
    print(f"\nscipy.linalg.eigh(A, B) は {type(exc).__name__} を送出する"
          f"（B が正定値でない）")

print("\nB を B + eps*I と正則化しても最大・最小固有値は eps^{-1} の速さで発散する")
reg_rows = []
for e in [1e-1, 1e-2, 1e-3, 1e-4]:
    lam_e = eigh(A2, B2 + e * np.eye(3), eigvals_only=True)
    reg_rows.append((e, float(lam_e.min()), float(lam_e.max())))
    print(f"  eps={e:.0e}   最小 {lam_e.min():+.4g}   最大 {lam_e.max():+.4g}")

A3, B3 = np.diag([2.0, 3.0, 0.0]), np.diag([1.0, 2.0, 0.0])   # Ker B = Ker A の場合
rng = np.random.default_rng(0)
Wr = rng.standard_normal((3, 2000))
num = np.einsum("ij,jk,ki->i", Wr.T, A3, Wr)
den = np.einsum("ij,jk,ki->i", Wr.T, B3, Wr)
rr = num / den
print(f"\nKer B ⊂ Ker A の例（A=diag(2,3,0), B=diag(1,2,0)）：乱数 2000 本の R の範囲 "
      f"[{rr.min():.6f}, {rr.max():.6f}]")
print("  → (Ker B)^perp 上の一般化固有値 3/2 と 2/1 の間に収まり、有界である")

fig, ax = plt.subplots(figsize=(7.0, 3.8))
styles = [("o-", C["red"]), ("s-", C["blue"]), ("^-", C["green"]), ("v-", C["orange"])]
for (name, vals), (mk, col) in zip(sing_vals.items(), styles):
    ax.loglog(ts, np.abs(vals), mk, color=col, ms=5, label=name)
ax.set_xlabel("t")
ax.set_ylabel(L(r"$|\mathcal{R}(w_0+t w_1)|$", r"$|\mathcal{R}(w_0+t w_1)|$"))
ax.set_title(L("$B$ が特異なとき Rayleigh 商は発散しうる（命題 6.4）",
               "Rayleigh quotient may blow up when $B$ is singular"))
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

出力のとおり三つの場合が分かれる。
(i) $\boldsymbol{w}_0=\boldsymbol{e}_2$（$\boldsymbol{w}_0^\top\boldsymbol{A}\boldsymbol{w}_0=+1$）では
$t=10^{-1},10^{-2},10^{-3}$ で
102、
1e+04、
1e+06 と $t^{-2}$ の速さで $+\infty$ へ、
(ii) $\boldsymbol{w}_0=\boldsymbol{e}_3$（$-1$）では同じ速さで $-\infty$ へ向かう。
(iii) $\boldsymbol{w}_0=\boldsymbol{e}_2+\boldsymbol{e}_3$（二次形式 0）では、経路 $\boldsymbol{w}_1=\boldsymbol{e}_1$ なら
$\mathcal{R}=2.000000$ で一定なのに、
経路 $\boldsymbol{w}_1=\boldsymbol{e}_1+\boldsymbol{e}_2$ では
23、
203、
2003 と発散する——
同じ $\boldsymbol{w}_0$ に近づいているのに極限が違うので、値を割り当てようがない。

`scipy.linalg.eigh(A, B)` はこの $\boldsymbol{B}$ に対して例外を送出する。ライブラリも定義 6.1 の
「$\boldsymbol{B}$ は正定値」という仮定を要求している。実務の処方は
$\boldsymbol{B}\to\boldsymbol{B}+\epsilon\boldsymbol{I}$ だが、これは問題を有限にするだけで、
$\epsilon=1e-01,1e-02,1e-03,1e-04$ に対し
最大固有値が 10、100、1000、10000、
最小固有値が -10、-100、-1000、-10000 と
$\epsilon^{-1}$ の速さで両側に発散する。

有界性の正しい判定条件は $\operatorname{Ker}\boldsymbol{B}\subset\operatorname{Ker}\boldsymbol{A}$ である
（命題 6.4 の最後）。$\boldsymbol{A}=\operatorname{diag}(2,3,0)$、$\boldsymbol{B}=\operatorname{diag}(1,2,0)$ で
乱数 2000 本を試すと $\mathcal{R}$ の範囲は $[1.500000,\ 2.000000]$ で、
$(\operatorname{Ker}\boldsymbol{B})^\perp$ 上の一般化固有値 $3/2$ と $2/1$ にきちんと収まる。

<a name="62"></a>
## 6.2 LDA と PCA は違う方向を選ぶ

本文 6.3 節。LDA は $\boldsymbol{A}=\boldsymbol{S}_B$、$\boldsymbol{B}=\boldsymbol{S}_W$ の Rayleigh 商
$J(\boldsymbol{w})=\boldsymbol{w}^\top\boldsymbol{S}_B\boldsymbol{w}/\boldsymbol{w}^\top\boldsymbol{S}_W\boldsymbol{w}$（式 (6.9)）を最大化し、
2 クラスなら閉形式 $\boldsymbol{w}^\star\propto\boldsymbol{S}_W^{-1}(\boldsymbol{\mu}_1-\boldsymbol{\mu}_2)$（定理 6.12、式 (6.10)）で
書ける。PCA は全散布 $\boldsymbol{S}_T=\boldsymbol{S}_W+\boldsymbol{S}_B$ の最大固有ベクトルを取る。
例 6.15 と図6.1 の状況——二クラスが相関の強い共通共分散をもち、
分散最大の方向がクラスを結ぶ方向ではない——を作って比べる。

In [ ]:
def scatter_matrices(X_, y_):
    """クラス内・クラス間散布行列（式 (6.8)）。X は d×n（列がサンプル）。"""
    d_, n_ = X_.shape
    mu_all = X_.mean(1)
    Sw = np.zeros((d_, d_)); Sb = np.zeros((d_, d_))
    for c_ in np.unique(y_):
        Xc_ = X_[:, y_ == c_]
        mu_c = Xc_.mean(1)
        Z_ = Xc_ - mu_c[:, None]
        Sw += Z_ @ Z_.T / n_
        Sb += Xc_.shape[1] * np.outer(mu_c - mu_all, mu_c - mu_all) / n_
    return Sw, Sb


rng = np.random.default_rng(0)
n_per = 300
theta_rot = np.pi / 4
Rrot = np.array([[np.cos(theta_rot), -np.sin(theta_rot)],
                 [np.sin(theta_rot), np.cos(theta_rot)]])
Sw_true = Rrot @ np.diag([4.0, 0.16]) @ Rrot.T        # (1,1) 方向に細長い共通共分散
Lh = np.linalg.cholesky(Sw_true)
mu_a = np.array([1.1, -1.1]); mu_b = -mu_a            # 平均差は (1,-1) 方向
X = np.hstack([Lh @ rng.standard_normal((2, n_per)) + mu_a[:, None],
               Lh @ rng.standard_normal((2, n_per)) + mu_b[:, None]])  # 2×600
y = np.r_[np.zeros(n_per, int), np.ones(n_per, int)]

Sw, Sb = scatter_matrices(X, y)
St = Sw + Sb
mu1 = X[:, y == 0].mean(1); mu2 = X[:, y == 1].mean(1)
w_lda = np.linalg.solve(Sw, mu1 - mu2)                # 定理 6.12：S_W^{-1}(mu1-mu2)
w_lda /= np.linalg.norm(w_lda)
ev_t, U_t = np.linalg.eigh(St)
w_pca = U_t[:, -1]                                    # 全散布の最大固有ベクトル
print("S_T = S_W + S_B か:", np.allclose(St, np.cov(X, bias=True)))
print(f"LDA の方向 {np.round(w_lda, 4)}   PCA の方向 {np.round(w_pca, 4)}"
      f"   なす角 {np.degrees(np.arccos(abs(w_lda @ w_pca))):.1f} 度")


def fisher_J(w_):
    return float((w_ @ Sb @ w_) / (w_ @ Sw @ w_))


J_lda, J_pca = fisher_J(w_lda), fisher_J(w_pca)
print(f"判別基準 J(w)（式 (6.9)）：LDA 方向 {J_lda:.4f}   PCA 方向 {J_pca:.4f}"
      f"   （比 {J_lda/J_pca:.0f} 倍）")

proj = {"LDA": X.T @ w_lda, "PCA": X.T @ w_pca}       # 射影は X^T w（n 個のスカラー）
overlap = {}
for k_, p_ in proj.items():
    lo, hi = p_.min(), p_.max()
    bins = np.linspace(lo, hi, 60)
    h0, _ = np.histogram(p_[y == 0], bins=bins, density=True)
    h1, _ = np.histogram(p_[y == 1], bins=bins, density=True)
    overlap[k_] = float(np.sum(np.minimum(h0, h1)) * (bins[1] - bins[0]))
    print(f"{k_} 方向に射影したときの二クラスの重なり = {overlap[k_]:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
axes[0].scatter(X[0, y == 0], X[1, y == 0], s=6, alpha=0.5, color=C["blue"])
axes[0].scatter(X[0, y == 1], X[1, y == 1], s=6, alpha=0.5, color=C["orange"])
for w_, col, nm in [(w_pca, C["red"], "PCA"), (w_lda, C["green"], "LDA")]:
    axes[0].annotate("", xy=4.5 * w_, xytext=-4.5 * w_,
                     arrowprops=dict(arrowstyle="<->", color=col, lw=2))
    axes[0].text(4.9 * w_[0], 4.9 * w_[1], nm, color=col, fontsize=11)
axes[0].set_aspect("equal"); axes[0].set_xlim(-7, 7); axes[0].set_ylim(-7, 7)
axes[0].set_title(L("同じデータ、違う方向（図6.1）", "same data, different directions"))
for ax, k_ in zip(axes[1:], ["PCA", "LDA"]):
    ax.hist(proj[k_][y == 0], bins=40, alpha=0.6, color=C["blue"], density=True)
    ax.hist(proj[k_][y == 1], bins=40, alpha=0.6, color=C["orange"], density=True)
    ax.set_title(f"{k_} " + L(f"方向への射影（重なり {overlap[k_]:.2f}）",
                              f"projection (overlap {overlap[k_]:.2f})"))
    ax.set_xlabel(L("射影値", "projected value"))
fig.tight_layout()
plt.show()

共通共分散を $(1,1)$ 方向に細長く（固有値 4 と 0.16）、
クラス平均差を $(1,-1)$ 方向に置いた。LDA が選ぶのは
$(0.712,\ -0.702)$、PCA が選ぶのは $(0.692,\ 0.722)$ で、
両者のなす角は 89.2 度——ほぼ直交である。

判別基準の値は LDA 方向で $J=15.7328$、PCA 方向で $J=0.0003$、
比にして 60499 倍の差がある。射影後のヒストグラムの重なりも
PCA 方向で 0.757、LDA 方向で 0.000 で、
**PCA の選ぶ方向に射影した時点で判別情報はほぼ失われている**。
分散の大きさは情報の多さを意味しない。ラベルがあるならラベルに関係する方向を選ぶべきである
（ただしこれは PCA の欠陥ではなく、教師なし手法に判別を期待するほうが誤りである）。

### 多クラス：LDA が取り出せる方向は $C-1$ 本まで

命題 6.14 は $\operatorname{rank}\boldsymbol{S}_B\le C-1$ を言う。iris（$d=4$、$n=150$、$C=3$）で
一般化固有値問題 $\boldsymbol{S}_B\boldsymbol{w}=\lambda\boldsymbol{S}_W\boldsymbol{w}$ を解き、非零固有値が 2 個しかないことと、
2 次元に落としたときの見え方を PCA と比べる。

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
Xi = iris.data.T                     # 4×150 に転置：本講義の規約は列がサンプル
yi = iris.target
n_class = len(np.unique(yi))
Swi, Sbi = scatter_matrices(Xi, yi)
print(f"iris: d={Xi.shape[0]}, n={Xi.shape[1]}, クラス数 C={n_class}")
print(f"rank(S_B) = {np.linalg.matrix_rank(Sbi)}（命題 6.14 の上限 C-1 = {n_class-1}）")

lam_i, W_i = eigh(Sbi, Swi)          # S_B w = lambda S_W w
lam_i, W_i = lam_i[::-1], W_i[:, ::-1]
print("一般化固有値（降順）:", np.array2string(lam_i, precision=4, suppress_small=True))
Z_lda = W_i[:, :2].T @ Xi            # LDA スコア（2×150）
Xc_i = Xi - Xi.mean(1, keepdims=True)
U_i = np.linalg.svd(Xc_i, full_matrices=False)[0]
Z_pca = U_i[:, :2].T @ Xc_i          # PCA スコア（2×150）

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8))
for ax, Z_, ttl in [(axes[0], Z_pca, "PCA"), (axes[1], Z_lda, "LDA")]:
    for c_, col in zip(range(n_class), [C["blue"], C["orange"], C["green"]]):
        ax.scatter(Z_[0, yi == c_], Z_[1, yi == c_], s=14, alpha=0.75, color=col,
                   label=iris.target_names[c_])
    ax.set_xlabel(L("第1軸", "axis 1")); ax.set_ylabel(L("第2軸", "axis 2"))
    ax.set_title(ttl)
axes[1].legend(fontsize=8)
fig.suptitle(L("iris を 2 次元に落とす：教師なし（PCA）と教師あり（LDA）",
               "iris in 2D: unsupervised (PCA) vs supervised (LDA)"))
fig.tight_layout()
plt.show()

一般化固有値は
$(32.1919,\ 0.2854,\ -1.63e-15,\ -7.11e-15)$ で、
非零は 2 個＝$C-1=2$ 本。命題 6.14 のとおりである
（$\operatorname{rank}\boldsymbol{S}_B$ も 2 と出る）。
図では PCA も iris の三品種をかなり分けるが、これは品種差が最大分散方向とたまたま
揃っているからで、LDA のほうがクラス内のばらつきに対する分離が明瞭である。
$C=2$ なら 1 本、$C=10$ でも 9 本しか取れないので、クラス数が少ないときは
LDA で落とせる次元に厳しい上限がある。

<a name="63"></a>
## 6.3 正準相関分析と階数不足による退化

本文 6.2 節。CCA は式 (6.5) のブロック行列
$\boldsymbol{A}=\begin{pmatrix}\boldsymbol{0}&\boldsymbol{\Sigma}_{xy}\\ \boldsymbol{\Sigma}_{xy}^\top&\boldsymbol{0}\end{pmatrix}$、
$\boldsymbol{B}=\begin{pmatrix}\boldsymbol{\Sigma}_{xx}&\boldsymbol{0}\\ \boldsymbol{0}&\boldsymbol{\Sigma}_{yy}\end{pmatrix}$ の
一般化固有値問題であり（命題 6.7）、実装上は
$\boldsymbol{M}=\boldsymbol{\Sigma}_{xx}^{-1/2}\boldsymbol{\Sigma}_{xy}\boldsymbol{\Sigma}_{yy}^{-1/2}$ の SVD で解くほうが安定である
（定理 6.8）。両者が一致することを確かめたうえで、
**階数が足りると $\rho_1=1$ に退化する**（式 (6.7)：$r_X+r_Y>n-1$）ことを実験で見る。

In [ ]:
from scipy.linalg import svd


def isqrt(M):
    """M^{-1/2}（M は正定値対称）。"""
    w_, U_ = np.linalg.eigh(M)
    return U_ @ np.diag(w_**-0.5) @ U_.T


rng = np.random.default_rng(0)
n_c, p_c, q_c = 500, 4, 3
Zl = rng.standard_normal((2, n_c))               # 共通の潜在因子 2 本（2×n）
Xcca = rng.standard_normal((p_c, 2)) @ Zl + 0.5 * rng.standard_normal((p_c, n_c))
Ycca = rng.standard_normal((q_c, 2)) @ Zl + 0.5 * rng.standard_normal((q_c, n_c))
Xcca -= Xcca.mean(1, keepdims=True)              # サンプル方向（列方向）に中心化
Ycca -= Ycca.mean(1, keepdims=True)
Sxx = Xcca @ Xcca.T / n_c + 1e-8 * np.eye(p_c)   # p×p
Syy = Ycca @ Ycca.T / n_c + 1e-8 * np.eye(q_c)   # q×q
Sxy = Xcca @ Ycca.T / n_c                        # p×q
Acca = np.block([[np.zeros((p_c, p_c)), Sxy], [Sxy.T, np.zeros((q_c, q_c))]])
Bcca = np.block([[Sxx, np.zeros((p_c, q_c))], [np.zeros((q_c, p_c)), Syy]])
rho_gev = np.sort(eigh(Acca, Bcca, eigvals_only=True))[::-1][:min(p_c, q_c)]
rho_svd = svd(isqrt(Sxx) @ Sxy @ isqrt(Syy), compute_uv=False)
print("一般化固有値（式 (6.5)）:", np.round(rho_gev, 6))
print("SVD の特異値（定理 6.8）:", np.round(rho_svd, 6))
print("一致:", np.allclose(rho_gev, rho_svd))


def rho1(Xa, Ya, eps=1e-10):
    """第一正準相関を SVD で求める。"""
    Xa = Xa - Xa.mean(1, keepdims=True); Ya = Ya - Ya.mean(1, keepdims=True)
    p_, q_ = Xa.shape[0], Ya.shape[0]
    n_ = Xa.shape[1]
    M = (isqrt(Xa @ Xa.T / n_ + eps * np.eye(p_)) @ (Xa @ Ya.T / n_)
         @ isqrt(Ya @ Ya.T / n_ + eps * np.eye(q_)))
    return float(svd(M, compute_uv=False)[0])


n_d, n_rep = 10, 200
r2 = np.random.default_rng(1)
deg = {}
for tag, rx, ry in [("r_X=5, r_Y=5（和 10 > n-1=9）", 5, 5),
                    ("r_X=5, r_Y=4（和  9 = n-1）", 5, 4)]:
    vals = []
    for _ in range(n_rep):
        Xa = r2.standard_normal((5, n_d))
        Ya = r2.standard_normal((ry, n_d))
        if ry < 5:
            Ya = np.vstack([Ya, np.zeros((5 - ry, n_d))])   # 階数を落とす
        vals.append(rho1(Xa, Ya))
    vals = np.array(vals)
    deg[tag] = vals
    print(f"{tag}: rho_1 の平均 {vals.mean():.6f}、最小 {vals.min():.6f}、"
          f"1.0 と見なせた回数（誤差 1e-6）{int(np.sum(vals > 1 - 1e-6))}/{n_rep}")

Xr = np.outer(rng.standard_normal(5), rng.standard_normal(n_d))   # 5 行が同一方向
rho_rank1 = rho1(Xr, r2.standard_normal((5, n_d)))
print(f"X の 5 行がすべて同じ方向（r_X=1）なら p=q=5, n=10 でも rho_1 = {rho_rank1:.4f}")

epss = np.logspace(-8, 0, 17)
Xa = r2.standard_normal((5, n_d)); Ya = r2.standard_normal((5, n_d))
rho_reg = [rho1(Xa, Ya, eps=e) for e in epss]
print(f"正則化 eps=1e-8 → rho_1={rho_reg[0]:.6f}、eps=1e-2 → "
      f"{rho1(Xa, Ya, eps=1e-2):.6f}、eps=1 → {rho_reg[-1]:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
for (tag, vals), col in zip(deg.items(), [C["red"], C["blue"]]):
    axes[0].hist(vals, bins=30, alpha=0.6, color=col, label=tag)
axes[0].set_xlabel(r"$\rho_1$"); axes[0].set_ylabel(L("回数", "count"))
axes[0].set_title(L("階数が足りると $\\rho_1$ が 1 に退化する",
                    r"rank deficiency degenerates $\rho_1$"))
axes[0].legend(fontsize=8)
axes[1].semilogx(epss, rho_reg, "o-", color=C["blue"])
axes[1].axhline(1.0, ls=":", color=C["red"])
axes[1].set_xlabel(r"$\epsilon$")
axes[1].set_ylabel(r"$\rho_1$")
axes[1].set_title(L("正則化は $\\rho_1<1$ を保証する（p=q=5, n=10）",
                    r"regularization guarantees $\rho_1<1$"))
fig.tight_layout()
plt.show()

潜在因子 2 本から作った $p=4$、$q=3$、$n=500$ のデータでは、
ブロック行列の一般化固有値も $\boldsymbol{M}$ の特異値も
$(0.9503,\ 0.2842,\ 0.0190)$ で一致する。
上位 2 本が大きく第 3 本がノイズ水準——CCA が共通の潜在構造の本数を検出している
（値そのものは乱数の実現に依存するので、見るべきはこの構造である）。

退化のほうは式 (6.7) の境界が鋭いことが確認できる。$n=10$ で 200 回試すと、
$r_X=r_Y=5$（和 $10>n-1=9$）では
196/200 回すべてで $\rho_1=1$ になるのに対し、
$r_X=5,\ r_Y=4$（和がちょうど $9$）では平均
0.9789 と高いものの $\rho_1=1$ は一度も起こらない。
これは**階数の条件であって次元の条件ではない**：$\boldsymbol{X}$ の 5 行がすべて同じ方向
（$r_X=1$）なら $p=q=5$、$n=10$ でも $\rho_1=0.4746$ にとどまる。

$\rho_1=1$ は「強い関係の発見」ではなく次元不足の合図かもしれない、というのが本文の警告である。
対策の正則化 $\boldsymbol{\Sigma}_{xx}+\epsilon\boldsymbol{I}$、$\boldsymbol{\Sigma}_{yy}+\epsilon\boldsymbol{I}$ は
$\rho_1<1$ を保証し、実測でも $\epsilon=10^{-8}$ の 1.0000 から
$\epsilon=1$ の 0.4107 まで単調に下がる。

<a name="64"></a>
## 6.4 古典的 MDS ＝ 双対 PCA

本文 6.4 節。距離しか手元にないとき、二重中心化
$\boldsymbol{B}=-\frac12\boldsymbol{H}\boldsymbol{D}^{(2)}\boldsymbol{H}$（式 (6.11)）の固有分解から座標を復元するのが
古典的 MDS（式 (6.12)）である。ユークリッド距離から出発すれば
$\boldsymbol{B}=\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}}$（式 (6.14)）で、これは $n\times n$ のグラム側、
すなわち**双対 PCA** そのものである（定理 6.20）。

順に、(a) 講義ノートのリスト（`lst:ch5-mds`）で PCA との一致、
(b) 完全埋め込み＋PCA という読み方（定理 6.21）と距離を縮める性質（系 6.22）、
(c) $\boldsymbol{B}$ が半正定値でない例（例 6.19 の四点輪 $C_4$ と、
距離の単調変換 $D_{ij}\mapsto D_{ij}^{1.4}$）を確かめる。

In [ ]:
def double_center(D2_):
    """二重中心化 B = -1/2 H D^(2) H（式 (6.11)）。"""
    n_ = D2_.shape[0]
    H_ = np.eye(n_) - np.ones((n_, n_)) / n_
    return -0.5 * H_ @ D2_ @ H_


def sq_dists(Z_):
    """列がサンプルの座標行列から二乗距離行列（n×n）を作る。"""
    return ((Z_[:, :, None] - Z_[:, None, :])**2).sum(0)   # 変数軸(0)で和


def cmds(D2_, k):
    """古典的 MDS（式 (6.12)）。正の固有値の上位 k 個だけを使う。"""
    lam_, V_ = np.linalg.eigh(double_center(D2_))
    idx = np.argsort(lam_)[::-1][:k]
    return (V_[:, idx] * np.sqrt(np.maximum(lam_[idx], 0))).T, np.sort(lam_)[::-1]


rng = np.random.default_rng(3)
n_m, d_m = 6, 4
Xm = rng.standard_normal((d_m, n_m))                  # d×n：列がサンプル
Hm = np.eye(n_m) - np.ones((n_m, n_m)) / n_m
Xcm = Xm @ Hm                                         # 中心化は右から H
Bm = double_center(sq_dists(Xcm))
print("B = Xc^T Xc か:", np.allclose(Bm, Xcm.T @ Xcm))   # グラム行列は n×n
Z_mds, _ = cmds(sq_dists(Xcm), 2)
Um = np.linalg.svd(Xcm, full_matrices=False)[0]       # 主成分方向は U の列
Z_pca2 = Um[:, :2].T @ Xcm                            # PCA スコア（2×n）
print("符号を除いて一致:", np.allclose(np.abs(Z_mds), np.abs(Z_pca2)))
print("MDS 第1列:", np.round(Z_mds[:, 0], 4), " PCA 第1列:", np.round(Z_pca2[:, 0], 4))

Zt = np.diag([3.0, 2.0, 1.0, 0.4]) @ np.random.default_rng(3).standard_normal((4, 14))
D_true = np.sqrt(sq_dists(Zt))                        # 14 点のユークリッド距離
lam_e = np.sort(np.linalg.eigvalsh(double_center(D_true**2)))[::-1]
r_pos = int(np.sum(lam_e > 1e-8))
print(f"\n14 点のユークリッド距離：B の正の固有値 {r_pos} 個 "
      f"{np.round(lam_e[:4], 1)}、最小固有値 {lam_e[-1]:.2e}")
Y_full, _ = cmds(D_true**2, r_pos)
print(f"完全埋め込み Y（{Y_full.shape[0]}×{Y_full.shape[1]}）の距離再現誤差 "
      f"{np.abs(np.sqrt(sq_dists(Y_full)) - D_true).max():.2e}")
print(f"Y の標本共分散の非対角の最大 {np.abs(Y_full @ Y_full.T / 14 - np.diag(np.diag(Y_full @ Y_full.T / 14))).max():.2e}"
      f"（対角は lambda_j/n に一致：{np.allclose(np.diag(Y_full @ Y_full.T / 14), lam_e[:r_pos]/14)}）")

iu = np.triu_indices(14, 1)
print("\n系 6.22（距離を縮める）と累積寄与率の一致（ユークリッドの場合）")
contr = []
for k in [1, 2, 3]:
    Zk, _ = cmds(D_true**2, k)
    dk = np.sqrt(sq_dists(Zk))[iu]; d0 = D_true[iu]
    contr.append((k, float((dk**2).sum() / (d0**2).sum()),
                  float(lam_e[:k].sum() / lam_e[:r_pos].sum()), int(np.sum(dk > d0 + 1e-9))))
    print(f"  k={k}: 二乗和の保存率 {contr[-1][1]:.4f}  累積寄与率 {contr[-1][2]:.4f}"
          f"  D_ij を超えた対 {contr[-1][3]}/91")

D_c4 = np.array([[0, 1, 2, 1], [1, 0, 1, 2], [2, 1, 0, 1], [1, 2, 1, 0]], float)
lam_c4 = np.sort(np.linalg.eigvalsh(double_center(D_c4**2)))[::-1]
print(f"\n例 6.19：四点輪 C_4 の最短路距離では B の固有値 {np.round(lam_c4, 4)}"
      f" → 負の固有値があるのでユークリッドではない（定理 6.18）")

D_ne = D_true**1.4                                    # 単調変換で非ユークリッドにする
lam_ne = np.sort(np.linalg.eigvalsh(double_center(D_ne**2)))[::-1]
n_neg = int(np.sum(lam_ne < -1e-8))
Y_ne, _ = cmds(D_ne**2, int(np.sum(lam_ne > 1e-8)))
print(f"\nD^1.4（非ユークリッド）：負の固有値 {n_neg} 個、最小 {lam_ne[-1]:.1f}、"
      f"正の固有値だけで作った Y の距離再現誤差 {np.abs(np.sqrt(sq_dists(Y_ne)) - D_ne).max():.2f}")
exceed = []
for k in [2, 3]:
    Zk, _ = cmds(D_ne**2, k)
    dk = np.sqrt(sq_dists(Zk))[iu]
    exceed.append((k, int(np.sum(dk > D_ne[iu] + 1e-9))))
    print(f"  k={k}: D_ij を超えた対 {exceed[-1][1]}/91 → 系 6.22 の単調性は失われる")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8))
Z2, _ = cmds(D_true**2, 2)
Xc14 = Zt - Zt.mean(1, keepdims=True)
U14 = np.linalg.svd(Xc14, full_matrices=False)[0]
Zp2 = U14[:, :2].T @ Xc14
axes[0].scatter(Z2[0], Z2[1], s=70, facecolors="none", edgecolors=C["blue"],
                label=L("古典的 MDS", "classical MDS"))
axes[0].scatter(Zp2[0], Zp2[1], s=18, color=C["red"], label=L("PCA スコア", "PCA scores"))
axes[0].set_title(L("距離だけから復元した座標＝PCA スコア（定理 6.20）",
                    "coordinates from distances = PCA scores"), fontsize=10)
axes[0].legend(fontsize=8); axes[0].set_aspect("equal")
idxb = np.arange(1, 15)
axes[1].bar(idxb - 0.2, lam_e, 0.4, color=C["blue"], label=L("ユークリッド距離", "Euclidean"))
axes[1].bar(idxb + 0.2, lam_ne / 30, 0.4, color=C["orange"],
            label=L(r"$D^{1.4}$（1/30 に縮尺）", r"$D^{1.4}$ (scaled 1/30)"))
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlabel(L("固有値の順位", "index")); axes[1].set_ylabel(L("固有値", "eigenvalue"))
axes[1].set_title(L("非ユークリッドだと負の固有値が出る", "non-Euclidean gives negatives"),
                  fontsize=10)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

(a) $n=6$、$d=4$ で $\boldsymbol{B}=\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}}$ が成り立ち、
MDS 座標と PCA スコアは符号を除いて一致する（どちらの検証も `True`）。

(b) 14 点（真の次元 4）のユークリッド距離では $\boldsymbol{B}$ の正の固有値が 4 個
（252.1, 33.4, 16.5, 0.879）、最小固有値は
-2.0e-14 で負が現れない（定理 6.18）。完全埋め込み $\boldsymbol{Y}\in\mathbb{R}^{4\times14}$ は
距離を誤差 1.1e-14 で再現し、
その標本共分散は対角（対角成分は $\lambda_j/n$）——つまり $\boldsymbol{Y}$ の座標軸はすでに主軸で、
「$\boldsymbol{Y}$ の PCA は上位 $k$ 行を取り出すだけ」になる（定理 6.21）。
距離の二乗和の保存率は $k=1,2,3$ で
0.8325、0.9427、0.9971 と累積寄与率
0.8325、0.9427、0.9971 に厳密に一致し、
$D_{ij}$ を超えた対は 91 対中 0 対——系 6.22 の「距離を縮める側にしか誤らない」が確認できる。

(c) 四点輪 $C_4$ の最短路距離では $\boldsymbol{B}$ の固有値が
$(2,\ 2,\ 0,\ -1)$ となり、
負の固有値が現れる＝ユークリッド距離行列ではない（例 6.19）。
14 点の距離を $D_{ij}^{1.4}$ と単調変換した場合も負の固有値が 10 個
（最小 -467.2）現れ、正の固有値だけで作った埋め込みの距離再現誤差は
8.65 に達する。
このとき $k=2$ では 91 対中 80 対、$k=3$ では
91 対が $D_{ij}$ を**超える**。負の固有値を捨てた分「距離が足りない」ので
残った正の成分が引き伸ばされるためで、
**「MDS は距離を縮める」はユークリッドのときに限った性質**である。

<a name="65"></a>
## 6.5 スペクトラルクラスタリング

本文 6.5 節。ガウス類似度 $W_{ij}=\exp(-\|\boldsymbol{x}_i-\boldsymbol{x}_j\|^2/2\sigma^2)$ から
ラプラシアン $\boldsymbol{L}=\boldsymbol{D}-\boldsymbol{W}$ を作り、$\boldsymbol{L}\boldsymbol{w}=\lambda\boldsymbol{D}\boldsymbol{w}$
（$\boldsymbol{A}=\boldsymbol{L}$、$\boldsymbol{B}=\boldsymbol{D}$、ただし**最小化**）を解く。これは NCut の緩和であり
（命題 6.31、式 (6.27)）、第 2 固有ベクトル（Fiedler ベクトル）の符号で 2 分割する。
講義ノートのリスト（`lst:ch5-spectral`）と図6.2 の再現である。

**$\sigma$ の感度について**：本文には実測値の表があるが、
点数・ノイズ・乱数種が変われば値も変わる。以下の表はこのノートブックの設定
（各群 150 点、ノイズ 0.06、`default_rng(0)`）での実測値である。

In [ ]:
from sklearn.cluster import KMeans
from scipy.sparse.csgraph import connected_components

rng = np.random.default_rng(0)
t = np.linspace(0, np.pi, 150)
Xs = np.hstack([np.vstack([np.cos(t), np.sin(t)]),                # 上の三日月
                np.vstack([1 - np.cos(t), 0.5 - np.sin(t)])])     # 下の三日月
Xs = Xs + 0.06 * rng.standard_normal((2, 300))                    # 2×300：列がサンプル
truth = np.r_[np.zeros(150), np.ones(150)]
D2s = ((Xs[:, :, None] - Xs[:, None, :])**2).sum(0)               # 変数軸(0)で和


def spectral(sigma, n_ev=6):
    """ガウス類似度からラプラシアンを作り L w = lambda D w を解く（本文 6.5 節）。"""
    Wg = np.exp(-D2s / (2 * sigma**2)); np.fill_diagonal(Wg, 0.0)
    Dg = np.diag(Wg.sum(1))
    Lap = Dg - Wg                                     # 非正規化ラプラシアン（n×n）
    lam_, V_ = eigh(Lap, Dg)                          # 一般化固有値問題
    labels_ = (V_[:, 1] > 0).astype(int)              # Fiedler ベクトルの符号で 2 分割
    acc = max((labels_ == truth).mean(), (labels_ != truth).mean())
    cross = Wg[:150, 150:].max()                      # 月をまたぐ最大の重み
    n_zero = int(np.sum(lam_ < 1e-10))                # 0 固有値の個数（定理 6.29）
    n_cc = connected_components(Wg > 1e-10, directed=False)[0]   # 重みの閾値で数えた成分数
    return lam_[:n_ev], labels_, float(acc), float(cross), n_zero, n_cc


sigma0 = 0.18
lam_s, lab_s, acc_s, cross_s, _, _ = spectral(sigma0)
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xs.T)  # sklearn は n×d
acc_km = max((km == truth).mean(), (km != truth).mean())
print(f"sigma={sigma0}: lambda_1..4 = {np.round(lam_s[:4], 5)}")
print(f"  Fiedler ベクトルの符号による正解率 = {acc_s:.3f}"
      f"（300 点中 {int(round((1-acc_s)*300))} 点の誤り）")
print(f"  同じデータに k-means を直接当てた正解率 = {acc_km:.3f}")
print(f"  lambda_3/lambda_2 = {lam_s[2]/lam_s[1]:.2f}（固有値ギャップ）")

print("\nsigma 依存性（この点数・ノイズ・乱数種での実測値。設定が変われば値も変わる）")
print("  sigma   lambda_2     lambda_3    比       またぎ最大重み  0 固有値  成分数  正解率")
sig_rows = []
for sg in [0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    lam_g, _, acc_g, cross_g, nz, ncc = spectral(sg)
    sig_rows.append((sg, float(lam_g[1]), float(lam_g[2]), acc_g, cross_g, nz, ncc))
    ratio_g = f"{lam_g[2]/lam_g[1]:8.1f}" if lam_g[1] > 1e-12 else "       -"
    print(f"  {sg:.2f}   {lam_g[1]:.3e}  {lam_g[2]:.3e}  {ratio_g}"
          f"     {cross_g:.2e}     {nz:3d}    {ncc:3d}    {acc_g:.3f}")
nz01, ncc01 = sig_rows[0][5], sig_rows[0][6]
nz03, ncc03 = sig_rows[1][5], sig_rows[1][6]
print(f"\nsigma=0.01：0 固有値 {nz01} 個／連結成分 {ncc01} 個 → グラフが砕けて分割が壊れる")
print(f"sigma=0.03：0 固有値 {nz03} 個／連結成分 {ncc03} 個。lambda_2 = {sig_rows[1][1]:.1e} は")
print("  機械精度の 0 で、二つの月はすでに数値的に非連結である（定理 6.29 の重複固有値）。")
print("  この場合の「成功」は 0 固有空間の中でたまたま良い基底が返っただけで、当てにならない。")

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
for ax, lab, ttl in [(axes[0], km, f"k-means（{L('正解率', 'acc')} {acc_km:.3f}）"),
                     (axes[1], lab_s, f"{L('スペクトラル', 'spectral')}"
                                      f"（{L('正解率', 'acc')} {acc_s:.3f}）")]:
    ax.scatter(Xs[0, lab == 0], Xs[1, lab == 0], s=8, color=C["blue"])
    ax.scatter(Xs[0, lab == 1], Xs[1, lab == 1], s=8, color=C["orange"])
    ax.set_title(ttl); ax.set_aspect("equal")
axes[2].plot(np.arange(1, 7), lam_s, "o-", color=C["blue"])
axes[2].set_xlabel(L("順位", "index")); axes[2].set_ylabel(r"$\lambda_k$")
axes[2].set_title(L(r"$\lambda_2$ の後にギャップ", r"gap after $\lambda_2$"))
fig.suptitle(L(f"スペクトラルクラスタリング（図6.2、n=300、$\\sigma$={sigma0}）",
               f"spectral clustering (Fig. 6.2, n=300)"))
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 3.8))
sg_all = [r[0] for r in sig_rows]
ax.plot(sg_all, [r[3] for r in sig_rows], "o-", color=C["blue"], label=L("正解率", "accuracy"))
ax.set_xlabel(r"$\sigma$"); ax.set_ylabel(L("正解率", "accuracy")); ax.set_ylim(0.4, 1.05)
ax2 = ax.twinx()
ok = [r for r in sig_rows if r[1] > 1e-12]      # lambda_2 が機械精度 0 の点は除く
ax2.semilogy([r[0] for r in ok], [r[2] / r[1] for r in ok], "s--", color=C["red"],
             label=L(r"$\lambda_3/\lambda_2$（$\lambda_2>0$ のみ）",
                     r"$\lambda_3/\lambda_2$ (where $\lambda_2>0$)"))
ax2.set_ylabel(r"$\lambda_3/\lambda_2$"); ax2.grid(False)
ax.set_title(L(r"$\sigma$ の感度（$\sigma\leq0.03$ ではグラフが分断される）",
               r"sensitivity to $\sigma$"))
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=9, loc="lower left")
fig.tight_layout()
plt.show()

$\sigma=0.18$ での固有値は
$\lambda_{1..4}=(0.00000,\ 0.00463,\ 0.02061,\ 0.02189)$、
Fiedler ベクトルの符号による正解率は 0.997（300 点中 1 点の誤り）。
同じデータに $k$-means を直接当てると 0.743 にとどまる。
$k$-means が失敗するのはユークリッド距離で球状のクラスタを探すからで、
スペクトラル法が成功するのは、グラフを経由して「つながっているか」という
位相的な近さに問題をすり替えているからである。
$\lambda_2$ の直後に $\lambda_3/\lambda_2=4.5$ 倍のギャップが立つことが
「クラスタが 2 つ」という構造の指標である（固有値ギャップ法）。

$\sigma$ 依存性（このノートブックでの実測値）：

| $\sigma$ | $\lambda_2$ | $\lambda_3$ | $\lambda_3/\lambda_2$ | またぎ最大重み | 連結成分 | 正解率 |
|---|---|---|---|---|---|---|
| 0.01 | -3.08e-16 | -2.45e-16 | — | 5.19e-129 | 63 | 0.593 |
| 0.03 | 1.72e-16 | 2.02e-05 | — | 5.57e-15 | 3 | 0.997 |
| 0.05 | 5.10e-08 | 8.14e-04 | 15953.8 | 7.39e-06 | 1 | 1.000 |
| 0.10 | 1.92e-04 | 5.00e-03 | 26.0 | 5.21e-02 | 1 | 1.000 |
| 0.15 | 1.69e-03 | 1.31e-02 | 7.7 | 2.69e-01 | 1 | 0.997 |
| 0.20 | 7.69e-03 | 2.73e-02 | 3.6 | 4.78e-01 | 1 | 0.997 |
| 0.30 | 3.16e-02 | 8.56e-02 | 2.7 | 7.20e-01 | 1 | 0.863 |
| 0.50 | 1.29e-01 | 3.25e-01 | 2.5 | 8.89e-01 | 1 | 0.793 |

比 $\lambda_3/\lambda_2$ が大きい $\sigma$ ほど正解率が高い、という傾向が読める。
$\sigma$ が大きすぎると二つの月をまたぐ重みが上がり（$\sigma=0.5$ で最大
0.89）両者がつながりはじめて、正解率は 0.793 まで落ちる。

逆に $\sigma$ が小さすぎるとグラフが分断される。$\sigma=0.01$ では連結成分が
63 個に砕ける。定理 6.29 より 0 固有値の重複度は連結成分の数 63 に等しいはずだが、
数値的に $10^{-10}$ を下回る固有値は 31 個しか数えられない——
孤立点があると $\boldsymbol{D}$ の対角が極端に小さくなり、
$\boldsymbol{L}\boldsymbol{w}=\lambda\boldsymbol{D}\boldsymbol{w}$ の条件数が悪化するためで、
「いくつが 0 か」の判定自体が閾値に依存してしまう。
いずれにせよ 0 固有空間が何十次元もあるので、数値解法が返す「第二固有ベクトル」は
各成分の指示ベクトルの任意の線形結合でしかなく、正解率は 0.593 に落ちる。
$\sigma=0.03$ は境目で、またぎ重みが 5.6e-15、
$\lambda_2=1.7e-16$ は機械精度の 0——二つの月はすでに数値的に非連結であり、
重みを $10^{-10}$ で切って数えた連結成分は 3 個（月 2 つ＋孤立点 1 つ）である。
この $\sigma$ での正解率 0.997 は
0 固有空間の中でたまたま良い基底が返っただけで、当てにできる成功ではない。
正しい対処は、$\sigma$ をいじる前に 0 固有空間の次元（＝連結成分の数）を数え、
成分ごとに処理することである。

**注意**：本文 6.5 節にも $\sigma$ 依存性の実測表があるが、
点数・ノイズ・乱数種が変われば値も変わる。上の表はこのノートブックの設定での値であり、
たとえば連結成分の数は重みをどの閾値で 0 と見なすかにも依存する
（この実装では $W_{ij}>10^{-10}$ を辺と見なしている）。
この設定で安定して働くのは $\sigma\in[0.05,\ 0.20]$ 程度だが、
**単一の表から普遍的な閾値を読み取ってはいけない**、というのが本節の教訓である。

<a name="66"></a>
## 6.6 部分最小二乗法（PLS）

注意 6.32。主成分回帰は $\boldsymbol{y}$ を一切見ずに方向を決めるので、
$\boldsymbol{y}$ と関係の薄い方向に大きな分散が乗っていれば無駄な成分を選ぶ。
PLS は $\boldsymbol{\Sigma}_{xy}$ を見て
$\boldsymbol{\Sigma}_{xy}\boldsymbol{\Sigma}_{xy}^\top\boldsymbol{w}=\lambda\boldsymbol{w}$（式 (6.28)、
$\boldsymbol{A}=\boldsymbol{\Sigma}_{xy}\boldsymbol{\Sigma}_{xy}^\top$、$\boldsymbol{B}=\boldsymbol{I}_d$）を解く。
説明変数に相関があり、真の方向が最大分散方向でない設定で確かめる。

In [ ]:
from sklearn.cross_decomposition import PLSRegression

rng = np.random.default_rng(0)
d_p, n_p = 8, 300
Qp = np.linalg.qr(rng.standard_normal((d_p, d_p)))[0]      # 固定した直交行列
lam_p_true = np.array([9., 6., 4., 3., 2., 1.5, 1., 0.7])  # 変数間に相関がある
Xp = Qp @ (np.sqrt(lam_p_true)[:, None] * rng.standard_normal((d_p, n_p)))   # d×n
Xp -= Xp.mean(1, keepdims=True)
w_star = Qp[:, 5]                     # 真の方向：分散は 6 番目でしかない
yv = w_star @ Xp + 0.5 * rng.standard_normal(n_p)
yv -= yv.mean()

Sxy_p = (Xp @ yv[:, None]) / n_p                      # d×1
lam_p, Wp = np.linalg.eigh(Sxy_p @ Sxy_p.T)           # A = Sxy Sxy^T, B = I（式 (6.28)）
w_pls = Wp[:, -1]
pls = PLSRegression(n_components=1, scale=False).fit(Xp.T, yv)   # sklearn は n×d
w_sk = pls.x_weights_[:, 0]
w_pc1 = np.linalg.svd(Xp, full_matrices=False)[0][:, 0]
print(f"式 (6.28) の最大固有ベクトルと PLSRegression の第一重みの |cos| = "
      f"{abs(w_pls @ w_sk):.12f}")
print(f"真の方向との重なり |cos|：PLS {abs(w_pls @ w_star):.3f}   第1主成分 "
      f"{abs(w_pc1 @ w_star):.3f}")
print("→ y を見るかどうかがそのまま効く（注意 6.32）")

式 (6.28) の最大固有ベクトルは `scikit-learn` の `PLSRegression` の第一重みと
$|\cos|=1.000000000000$ で一致する。真の方向との重なりは
PLS が 0.957 なのに対し第 1 主成分は 0.034 で、
$\boldsymbol{y}$ を見るかどうかがそのまま効いている。
LDA が $\boldsymbol{y}$ をクラスラベルとして使うのに対し PLS は連続値の $\boldsymbol{y}$ を使う、
という違いだけであり、どちらも表6.1 の一行に収まる。

<a name="ex"></a>
## 演習

1. **($\star$) 一般化固有値の手計算。**
   $\boldsymbol{A}=\begin{pmatrix}3&1\\1&3\end{pmatrix}$、
   $\boldsymbol{B}=\begin{pmatrix}1&0\\0&4\end{pmatrix}$ の一般化固有値を
   $\det(\boldsymbol{A}-\lambda\boldsymbol{B})=0$ から求め、`eigh(A, B)` と突き合わせよ。
   $\boldsymbol{C}=\boldsymbol{B}^{-1/2}\boldsymbol{A}\boldsymbol{B}^{-1/2}$ を書き下し、$\operatorname{tr}\boldsymbol{C}$ と
   $\det\boldsymbol{C}$ で検算せよ（本文 演習 6.1）。
2. **($\star\star$) PCA と LDA が一致する場合。** クラス内共分散を
   $\operatorname{diag}(4,\ 0.25)$、平均を $(\pm1,0)$ に取ると、6.2 節と違って
   PCA と LDA が同じ方向を選ぶ。数値で確かめ、一致するかどうかが
   共分散の形で決まることを述べよ（本文 演習 6.6 の追加課題）。
3. **($\star\star\star$) 単調変換の強さと単調性の破れ。** $D_{ij}^{p}$（$p=1,1.2,1.4,1.6$）
   について、$\boldsymbol{B}$ の負固有値の個数と $k=2$ の古典的 MDS で $D_{ij}$ を超える対の数を
   数え、$p$ が大きいほど系 6.22 の単調性が大きく破れることを確かめよ（本文 演習 6.8 (5)）。

In [ ]:
# 演習 1：A=[[3,1],[1,3]]、B=diag(1,4) の一般化固有値
# TODO: eigh(A, B) で解き、det(A - lam B) = 4 lam^2 - 15 lam + 8 = 0 の根
#       (15 ± sqrt(97))/8 と一致することを確かめる。
#       C = B^{-1/2} A B^{-1/2} を書き下し、tr C と det C も検算する

# 演習 2：クラス内共分散を diag(4, 0.25) に取り替えると PCA と LDA が一致する
# TODO: mu = (±1, 0)、S_W = diag(4, 0.25) で各クラス 300 点を生成し、
#       scatter_matrices で S_W, S_B を作って両方向のなす角と J を比べる

# 演習 3：非ユークリッドな非類似度での MDS
# TODO: D_true**1.2 と D_true**1.6 について double_center の負固有値の個数と
#       k=2 の古典的 MDS で D_ij を超える対の数を数え、変換が強いほど
#       単調性の破れが大きくなることを確かめる

<a name="sol"></a>
## 演習の解答

In [ ]:
# --- 演習 1 ---
Ae = np.array([[3., 1.], [1., 3.]]); Be = np.diag([1., 4.])
lam_e1, W_e1 = eigh(Ae, Be)
roots = np.sort([(15 - np.sqrt(97)) / 8, (15 + np.sqrt(97)) / 8])
Ce = np.diag([1., 0.5]) @ Ae @ np.diag([1., 0.5])          # B^{-1/2} = diag(1, 1/2)
print("演習1: eigh の固有値", np.round(lam_e1, 6), " 手計算 (15±sqrt(97))/8",
      np.round(roots, 6))
print(f"       C = {np.round(Ce, 4).tolist()}  tr C = {np.trace(Ce):.4f}(=15/4)"
      f"  det C = {np.linalg.det(Ce):.4f}(=2)")
print("       B-正規化した固有ベクトル:", np.round(W_e1[:, ::-1], 4).T.tolist())

# --- 演習 2 ---
rng_e = np.random.default_rng(1)
Sw_e = np.diag([4.0, 0.25])
mu_e = np.array([1.0, 0.0])
Xe = np.hstack([np.sqrt(Sw_e) @ rng_e.standard_normal((2, 300)) + mu_e[:, None],
                np.sqrt(Sw_e) @ rng_e.standard_normal((2, 300)) - mu_e[:, None]])
ye = np.r_[np.zeros(300, int), np.ones(300, int)]
Sw_e2, Sb_e2 = scatter_matrices(Xe, ye)
w_lda_e = np.linalg.solve(Sw_e2, Xe[:, ye == 0].mean(1) - Xe[:, ye == 1].mean(1))
w_lda_e /= np.linalg.norm(w_lda_e)
w_pca_e = np.linalg.eigh(Sw_e2 + Sb_e2)[1][:, -1]
ang_e = np.degrees(np.arccos(min(abs(w_lda_e @ w_pca_e), 1.0)))
J_l = (w_lda_e @ Sb_e2 @ w_lda_e) / (w_lda_e @ Sw_e2 @ w_lda_e)
J_p = (w_pca_e @ Sb_e2 @ w_pca_e) / (w_pca_e @ Sw_e2 @ w_pca_e)
print(f"\n演習2: S_W=diag(4,0.25) では LDA {np.round(w_lda_e,3)} と "
      f"PCA {np.round(w_pca_e,3)} のなす角 {ang_e:.1f} 度")
print(f"       J: LDA {J_l:.4f} / PCA {J_p:.4f} → 一致するかは共分散の形で決まる")

# --- 演習 3 ---
print("\n演習3: 単調変換の強さと単調性の破れ")
ex3_rows = []
for pw in [1.0, 1.2, 1.4, 1.6]:
    Dp_ = D_true**pw
    lam_p_ = np.sort(np.linalg.eigvalsh(double_center(Dp_**2)))[::-1]
    Zk_, _ = cmds(Dp_**2, 2)
    n_ex = int(np.sum(np.sqrt(sq_dists(Zk_))[iu] > Dp_[iu] + 1e-9))
    ex3_rows.append((pw, int(np.sum(lam_p_ < -1e-8)), float(lam_p_[-1]/lam_p_[0]), n_ex))
    print(f"  D^{pw}: 負の固有値 {ex3_rows[-1][1]:2d} 個  "
          f"最小/最大 = {ex3_rows[-1][2]:+.4f}  k=2 で D_ij を超えた対 {n_ex}/91")

**演習 1**：$\det(\boldsymbol{A}-\lambda\boldsymbol{B})=(3-\lambda)(3-4\lambda)-1=4\lambda^2-15\lambda+8=0$ より
$\lambda=(15\pm\sqrt{97})/8$、数値では $(0.643893,\ 3.106107)$ で
`eigh(A, B)` の出力と一致する。$\boldsymbol{B}^{-1/2}=\operatorname{diag}(1,1/2)$ なので
$\boldsymbol{C}=\begin{pmatrix}3&1/2\\1/2&3/4\end{pmatrix}$、
$\operatorname{tr}\boldsymbol{C}=3.7500=15/4$、
$\det\boldsymbol{C}=2.0000=2$ で検算できる。

**演習 2**：$\boldsymbol{S}_W=\operatorname{diag}(4,0.25)$ では LDA 方向と PCA 方向のなす角は
4.8 度で、$J$ も LDA 0.2367 / PCA 0.2366 とほぼ同じ。
6.2 節では両者がほぼ直交していたのに、ここでは一致する。
つまり**両者が一致するかどうかはクラス内共分散の形（最大分散の方向が
クラス平均差の方向と揃っているか）で決まる**のであって、
PCA が常に判別を壊すわけではない。

**演習 3**：$p$ を上げるほど非ユークリッド性が強まる。実測は

| $p$ | 負固有値の個数 | 最小/最大 | $k=2$ で $D_{ij}$ を超えた対 |
|---|---|---|---|
| 1.0 | 0 | -0.0000 | 0/91 |
| 1.2 | 9 | -0.1034 | 74/91 |
| 1.4 | 10 | -0.2030 | 80/91 |
| 1.6 | 10 | -0.2908 | 86/91 |

で、$p=1$（ユークリッド）では負固有値も超過対も 0、$p$ が大きいほど負の固有値の
比重が増え、超過する対も増える。系 6.22 の単調性は $\boldsymbol{B}\succeq0$ に依存しており、
それが崩れると結論も崩れる。距離の再現を優先するなら、
本文 6.4 節の計量 MDS（ストレス最小化）を $k$ 次元で直接回すほうが理にかなっている。